# Household Energy Consumption - Feature Engineering and Modeling

This notebook loads the cleaned hourly dataset, performs feature engineering (temporal features, lags, and rolling statistics), splits the data into training and test sets, and trains three models: ARIMA, Prophet, and XGBoost. The forecasts and feature importance scores are exported for evaluation.

## Setup and Library Imports

We import data manipulation, time-series forecasting, and machine learning libraries. Specifically:
- `statsmodels.tsa.arima.model.ARIMA` for statistical modeling.
- `prophet.Prophet` for trend-seasonality decomposed modeling.
- `xgboost.XGBRegressor` for gradient boosted machine learning modeling.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet
from xgboost import XGBRegressor

## Load the Cleaned Dataset

We load our preprocessed hourly energy consumption dataset.

In [2]:
hourly = pd.read_csv(
    r"C:\Users\Waleed Qamar\Downloads\energy-forecasting\data\household_power_consumption_cleaned.txt",
    parse_dates=["Datetime"],
    index_col="Datetime"
)

## Feature Engineering

We construct three categories of features:
1. **Temporal Features**: Extracting hour, day, month, weekday, and weekend indicator from the timestamp index to capture periodic cycles.
2. **Lag Features**: Shifts of 1 hour, 24 hours (1 day), and 168 hours (1 week) to capture auto-correlation.
3. **Rolling Statistics**: 24-hour rolling average to capture the recent local level of consumption.

In [3]:
# Temporal features
hourly["Hour"] = hourly.index.hour
hourly["Day"] = hourly.index.day
hourly["Month"] = hourly.index.month
hourly["Weekday"] = hourly.index.weekday
hourly["Weekend"] = (hourly["Weekday"] >= 5).astype(int)

# Lag features
hourly["Lag1"] = hourly["Global_active_power"].shift(1)
hourly["Lag24"] = hourly["Global_active_power"].shift(24)
hourly["Lag168"] = hourly["Global_active_power"].shift(168)

# Rolling features
hourly["Rolling24"] = hourly["Global_active_power"].rolling(24).mean()

### Handle Missing Values from Feature Generation

Generating lags and rolling statistics introduces missing rows (NaNs) at the beginning of the dataset (e.g. up to 168 hours). We drop these rows to ensure a clean, aligned training set.

In [4]:
hourly = hourly.dropna()

## Train/Test Split

We split our data sequentially: keeping all records except the last 500 hours for training (`train`), and reserving the last 500 hours for testing (`test`). A sequential split is required for time-series validation to prevent future information leakage into training.

In [5]:
train = hourly.iloc[:-500]
test = hourly.iloc[-500:]

print(f"Train size: {train.shape[0]}, Test size: {test.shape[0]}")

Train size: 32887, Test size: 500


## Model 1 — ARIMA (Autoregressive Integrated Moving Average)

ARIMA is a classic statistical method that models a series based on its own past values, differences, and forecast errors. We use order (5, 1, 2) and fit it on the primary target variable series.

In [6]:
train_series = train["Global_active_power"]
test_series = test["Global_active_power"]

arima_model = ARIMA(train_series, order=(5,1,2))
arima_fit = arima_model.fit()
arima_forecast = arima_fit.forecast(steps=len(test))

c:\Users\Waleed Qamar\Downloads\energy-forecasting\task3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Waleed Qamar\Downloads\energy-forecasting\task3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Waleed Qamar\Downloads\energy-forecasting\task3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Waleed Qamar\Downloads\energy-forecasting\task3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood 

## Model 2 — Prophet

Prophet is a time series forecasting library developed by Meta, designed for analyzing datasets with strong seasonal patterns and multiple history trends. We format the dataset with `ds` (datetime) and `y` (target) columns before fitting.

In [7]:
# Prepare Prophet datasets
prophet_train = train.reset_index()[["Datetime", "Global_active_power"]]
prophet_train.columns = ["ds", "y"]

prophet_test = test.reset_index()[["Datetime"]]
prophet_test.columns = ["ds"]

# Fit and forecast
prophet_model = Prophet()
prophet_model.fit(prophet_train)
prophet_forecast_res = prophet_model.predict(prophet_test)
prophet_forecast = prophet_forecast_res["yhat"].values

14:05:03 - cmdstanpy - INFO - Chain [1] start processing
14:05:12 - cmdstanpy - INFO - Chain [1] done processing


## Model 3 — XGBoost (Extreme Gradient Boosting)

XGBoost is an efficient supervised machine learning model. Unlike ARIMA and Prophet, XGBoost does not inherently understand time series sequences; instead, it relies on our engineered lag and temporal features to make predictions.

In [8]:
features = [
    "Hour",
    "Day",
    "Month",
    "Weekday",
    "Weekend",
    "Lag1",
    "Lag24",
    "Lag168",
    "Rolling24"
]

xgb_model = XGBRegressor()
xgb_model.fit(train[features], train["Global_active_power"])
xgb_forecast = xgb_model.predict(test[features])

## Export Model Predictions and Feature Importances

We aggregate actual target values and the predictions from all three models into a DataFrame and export it as `data/model_predictions.csv`. We also export the XGBoost feature importances as `data/xgboost_feature_importances.csv` to allow isolated evaluation.

In [9]:
predictions_df = pd.DataFrame(index=test.index)
predictions_df["Actual"] = test["Global_active_power"].values
predictions_df["ARIMA_Forecast"] = arima_forecast.values
predictions_df["Prophet_Forecast"] = prophet_forecast
predictions_df["XGBoost_Forecast"] = xgb_forecast

predictions_df.to_csv(r"C:\Users\Waleed Qamar\Downloads\energy-forecasting\data\model_predictions.csv")

importance = pd.Series(xgb_model.feature_importances_, index=features)
importance.to_csv(r"C:\Users\Waleed Qamar\Downloads\energy-forecasting\data\xgboost_feature_importances.csv")
print("Predictions and feature importances successfully exported!")

Predictions and feature importances successfully exported!
